# Pandas Part 2 — Phase 2 Cheatsheet
## Data Types and Transformations

### 1. Inspect Data Types
```python
df.dtypes                 # Data type of each column
df.info()                 # Column info, non-null counts, types, memory
df["Age"].dtype           # Data type of one column
```

Common types: `int64`, `float64`, `object`, `bool`, `string`, `datetime64[ns]`.

### 2. Convert Data Types
```python
df["Age"] = df["Age"].astype(int)          # Convert to integer
df["Age"] = df["Age"].astype(float)        # Convert to float
df["Name"] = df["Name"].astype(str)        # Convert to string
df["Score"] = pd.to_numeric(df["Score"])   # Convert to numeric
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"])    # Convert to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
```

- `errors="raise"`: raise an error for invalid values (default).
- `errors="coerce"`: invalid values become `NaN` or `NaT`.
- `NaN`: missing numeric value; `None`: Python null object; `NaT`: missing date/time.
- Use `df.isna()` to detect missing values.
- `astype(int)` can fail if values are invalid or missing.

### 3. Rename Columns
```python
df = df.rename(columns={"old_name": "new_name"})
df.rename(columns={"Age": "Employee_Age"}, inplace=True)
df.columns = df.columns.str.lower()       # Lowercase all names
```

### 4. Add and Update Columns
```python
df["Bonus"] = df["Salary"] * 0.10         # Add calculated column
df["Country"] = "India"                   # Add constant column
df["Total"] = df["Salary"] + df["Bonus"]  # Combine columns

df.loc[df["Salary"] >= 50000, "Active"] = True  # Conditional update
df["Age"] = df["Age"] + 1                # Update existing values
```

### 5. Delete Columns
```python
df = df.drop(columns=["Bonus"])           # Delete one column
df = df.drop(columns=["A", "B"])          # Delete multiple columns
df.drop(columns=["Bonus"], inplace=True)  # Modify original DataFrame
```

### 6. Replace Values — replace()
```python
df["City"] = df["City"].replace("Bombay", "Mumbai")

df["Grade"] = df["Grade"].replace({
    "A": "Excellent",
    "B": "Good",
    "C": "Average"
})

df["Score"] = df["Score"].replace("unknown", np.nan)
```

Use `replace()` to substitute specified values. Unmatched values generally remain unchanged.

### 7. Map Values — map()
```python
df["Grade_Label"] = df["Grade"].map({
    "A": "Excellent",
    "B": "Good",
    "C": "Average"
})
```

- `map()` is primarily used on a Series.
- Dictionary keys map to their corresponding values.
- Unmapped values become missing.
- Use `replace()` for substitutions and `map()` for Series value mapping.

### 8. Conditional Logic — np.where()
```python
import numpy as np

df["Salary_Level"] = np.where(
    df["Salary"] >= 50000,
    "High",
    "Low"
)
```

Syntax:
`np.where(condition, value_if_true, value_if_false)`

Use `np.where()` for two possible outcomes.

### 9. Multiple Conditions — np.select()
```python
conditions = [
    df["Salary"] >= 60000,
    df["Salary"] >= 45000,
    df["Salary"] < 45000
]

choices = ["High", "Medium", "Low"]

df["Salary_Category"] = np.select(
    conditions,
    choices,
    default="Unknown"
)
```

- Conditions and choices correspond by position.
- The first matching condition wins.
- Put higher-priority or more specific conditions first.
- `default` handles rows where no condition matches; without it, the default is `0`.

### 10. Important Reminders
1. `dtypes` inspects column types; `info()` gives a broader overview.
2. Use `pd.to_numeric()` for numeric strings and `pd.to_datetime()` for dates.
3. `errors="coerce"` converts invalid values into missing values; inspect them afterward.
4. `None`, `NaN` and `NaT` represent missing values in different contexts.
5. `rename()` and `drop()` return a new DataFrame by default.
6. Use `.loc[condition, column]` for conditional updates.
7. `replace()` substitutes values; dictionary-based `map()` transforms a Series and makes unmatched values missing.
8. Use `np.where()` for two outcomes and `np.select()` for multiple conditions.


In [205]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["Ayaan", "Sara", "Rahul", "Zoya"],
    "Age": ["21", "25", "30", "28"],
    "Salary": ["40000", "50000", "60000", "45000"],
    "Joining_Date": ["2024-01-15", "2023-06-20",
                     "2022-09-10", "2024-03-05"],
    "Active": [True, False, True, True]
})

df

,Name,Age,Salary,Joining_Date,Active
0,Ayaan,21,40000,2024-01-15,True
1,Sara,25,50000,2023-06-20,False
2,Rahul,30,60000,2022-09-10,True
3,Zoya,28,45000,2024-03-05,True


In [206]:
df.dtypes #This shows the data type of every column.

Name            object
Age             object
Salary          object
Joining_Date    object
Active            bool
dtype: object

In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Name          4 non-null      object
 1   Age           4 non-null      object
 2   Salary        4 non-null      object
 3   Joining_Date  4 non-null      object
 4   Active        4 non-null      bool  
dtypes: bool(1), object(4)
memory usage: 260.0+ bytes


>Convert data types

In [208]:
# astype() — convert a column's type
# Use astype() when the existing values are already compatible with the target type.
df["Age"] = df["Age"].astype(float)
df

,Name,Age,Salary,Joining_Date,Active
0,Ayaan,21.0,40000,2024-01-15,True
1,Sara,25.0,50000,2023-06-20,False
2,Rahul,30.0,60000,2022-09-10,True
3,Zoya,28.0,45000,2024-03-05,True


In [209]:
df["Salary"] = df["Salary"].astype(str)

In [210]:
# pd.to_numeric() — convert values to numbers
# This is particularly useful when a column contains numeric strings or messy values.
df["Salary"] = pd.to_numeric(df["Salary"])


In [211]:
# The key advantage is the errors parameter. it ll change any non compatible to null
numbers = pd.Series(["100", "200", "unknown","300"])
pd.to_numeric(numbers, errors="coerce")

0    100.0
1    200.0
2      NaN
3    300.0
dtype: float64

   Parameter	            Behavior
- errors="raise"	        Raises an error for invalid values; default
- errors="coerce"	        Converts invalid values to missing values
- errors="ignore"	        Legacy behavior; deprecated in recent Pandas versions

In [212]:
# pd.to_datetime() — convert strings to dates
df["Joining_Date"]= pd.to_datetime(df["Joining_Date"])
df.dtypes

Name                    object
Age                    float64
Salary                   int64
Joining_Date    datetime64[ns]
Active                    bool
dtype: object

In [213]:
#hanlding invalid values
series = pd.Series(["2025-10-01", "2026-09-23","Unknown"])
pd.to_datetime(series, errors="coerce")

0   2025-10-01
1   2026-09-23
2          NaT
dtype: datetime64[ns]

In [214]:
#convert to a specific datettime format that the string is in
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"], format='%Y-%m-%d')
df

,Name,Age,Salary,Joining_Date,Active
0,Ayaan,21.0,40000,2024-01-15,True
1,Sara,25.0,50000,2023-06-20,False
2,Rahul,30.0,60000,2022-09-10,True
3,Zoya,28.0,45000,2024-03-05,True


# Understand NaN, None and NaT
- These represent missing or unavailable values in different contexts.
- Value	                    Meaning
- NaN	Not a Number;           commonly used for missing numeric data
- None	                    Python's null object; Pandas recognizes it as missing in many columns
- NaT	                        Missing date/time value

Important distinction: NaN, None and NaT are not interchangeable Python objects, even though Pandas recognizes them as missing in typical cases. Use isna() to test missingness rather than comparing a value directly with NaN.

In [215]:
missing_df = pd.DataFrame({
    "Score": [90, np.nan, 80],
    "Name": ["Ayaan", None, "Sara"],
    "Date": pd.to_datetime(["2024-01-01", None, "2024-03-01"])
})

missing_df
missing_df.isna().sum()

Score    1
Name     1
Date     1
dtype: int64

In [216]:
# Rename columns using rename()
df = df.rename(columns={
    "Name":"Emp_name"
})
df
# rename() returns a new DataFrame by default. Use assignment or inplace=True if you want to retain the change.

,Emp_name,Age,Salary,Joining_Date,Active
0,Ayaan,21.0,40000,2024-01-15,True
1,Sara,25.0,50000,2023-06-20,False
2,Rahul,30.0,60000,2022-09-10,True
3,Zoya,28.0,45000,2024-03-05,True


In [217]:
df.rename(columns={
    "Age":"Emp_age",
    "Salary":"Emp_sal"
}, inplace=True)
df

,Emp_name,Emp_age,Emp_sal,Joining_Date,Active
0,Ayaan,21.0,40000,2024-01-15,True
1,Sara,25.0,50000,2023-06-20,False
2,Rahul,30.0,60000,2022-09-10,True
3,Zoya,28.0,45000,2024-03-05,True


In [218]:
#we can also standardize all the columns
df.columns = df.columns.str.lower()
df.columns


Index(['emp_name', 'emp_age', 'emp_sal', 'joining_date', 'active'], dtype='object')

>Add and update columns

In [219]:
# Add a new column
df["bonus"] = df["emp_sal"]*100

Update existing values
df["country"] = "India"
this will change the whole country columns values to India

In [220]:
#updating using loc
df.loc[df["emp_sal"] < 50000 , "active"] = True
df

,emp_name,emp_age,emp_sal,joining_date,active,bonus
0,Ayaan,21.0,40000,2024-01-15,True,4000000
1,Sara,25.0,50000,2023-06-20,False,5000000
2,Rahul,30.0,60000,2022-09-10,True,6000000
3,Zoya,28.0,45000,2024-03-05,True,4500000


In [221]:
#deleting rows columns using drop()
df = df.drop(columns=["bonus"])
df

,emp_name,emp_age,emp_sal,joining_date,active
0,Ayaan,21.0,40000,2024-01-15,True
1,Sara,25.0,50000,2023-06-20,False
2,Rahul,30.0,60000,2022-09-10,True
3,Zoya,28.0,45000,2024-03-05,True


In [222]:
df.drop(columns="active", inplace=True)
df

,emp_name,emp_age,emp_sal,joining_date
0,Ayaan,21.0,40000,2024-01-15
1,Sara,25.0,50000,2023-06-20
2,Rahul,30.0,60000,2022-09-10
3,Zoya,28.0,45000,2024-03-05


In [223]:
df["Active"] = [True, False, False, True]

>Replace values using replace() and map()


In [224]:
# replace() — substitute specific values
# Replace a value throughout a DataFrame:
df["Active"] = df["Active"].replace({
    True: "Yes",
    False:"No"
})
df

,emp_name,emp_age,emp_sal,joining_date,Active
0,Ayaan,21.0,40000,2024-01-15,Yes
1,Sara,25.0,50000,2023-06-20,No
2,Rahul,30.0,60000,2022-09-10,No
3,Zoya,28.0,45000,2024-03-05,Yes


In [225]:
series = pd.Series(["Good", "Average", "Bad"])
series.replace({
    "Good": "Pass",
    "Average":"Pass",
    "Bad":"Fail"
})
series

0       Good
1    Average
2        Bad
dtype: object

In [226]:
import numpy as np
series = pd.Series(["100", "unknown", "200", "300"])
series.replace({
    "unknown":np.nan
}, inplace=True)
series

0    100
1    NaN
2    200
3    300
dtype: object

In [227]:
# map() — map values to corresponding outputs

df["Department"] = df["emp_name"].map({ #this will create a new column
    "Ayaan" : "IT",
    "Sara" : "Finance",
    "Rahul":"IT"
}) #if we dont mention a column it ll keep the value as NULL or NaN
df

,emp_name,emp_age,emp_sal,joining_date,Active,Department
0,Ayaan,21.0,40000,2024-01-15,Yes,IT
1,Sara,25.0,50000,2023-06-20,No,Finance
2,Rahul,30.0,60000,2022-09-10,No,IT
3,Zoya,28.0,45000,2024-03-05,Yes,NaN


In [228]:
series = pd.Series(["A", "B", "C", "D"])
series.map({
    "A":"a",
    "B":"b"
}) #if we dont specify a column it ll still change it and keep the value as NAN

#NOTE: With dictionary-based map(), values that have no matching key become missing (NaN or an appropriate missing value for the dtype). Ensure your mapping covers all possible values if you don't want this.

0      a
1      b
2    NaN
3    NaN
dtype: object

> Conditional logic with NumPy

In [229]:
# np.where() — choose between two outcomes
# Suppose you want to classify employees based on salary:
df["sal_category"] = np.where( #this ll create a new column
    df["emp_sal"]>=50000,
    "high salary",
    "low salary"
)
df

,emp_name,emp_age,emp_sal,joining_date,Active,Department,sal_category
0,Ayaan,21.0,40000,2024-01-15,Yes,IT,low salary
1,Sara,25.0,50000,2023-06-20,No,Finance,high salary
2,Rahul,30.0,60000,2022-09-10,No,IT,high salary
3,Zoya,28.0,45000,2024-03-05,Yes,NaN,low salary


In [230]:
df["sal_category"] = np.where( # we can change values as well
    df["emp_sal"] >= 50000,
    "Highhhh",
    "Lowwwww"
)
df

,emp_name,emp_age,emp_sal,joining_date,Active,Department,sal_category
0,Ayaan,21.0,40000,2024-01-15,Yes,IT,Lowwwww
1,Sara,25.0,50000,2023-06-20,No,Finance,Highhhh
2,Rahul,30.0,60000,2022-09-10,No,IT,Highhhh
3,Zoya,28.0,45000,2024-03-05,Yes,NaN,Lowwwww


In [231]:
# np.select() — choose among multiple conditions
conditions = [
    df["emp_name"] == "Ayaan",
    df["emp_name"] == "Sara"
    ]
choices = ["Adnan", "Saira"]
df["emp_name"] = np.select(
    conditions,
    choices,
    default = "Unknown"
)
df


,emp_name,emp_age,emp_sal,joining_date,Active,Department,sal_category
0,Adnan,21.0,40000,2024-01-15,Yes,IT,Lowwwww
1,Saira,25.0,50000,2023-06-20,No,Finance,Highhhh
2,Unknown,30.0,60000,2022-09-10,No,IT,Highhhh
3,Unknown,28.0,45000,2024-03-05,Yes,NaN,Lowwwww
